<div class='alert alert-warning'>

# JupyterLite warning

- Running the **scikit-plots 0.5.dev0** examples in JupyterLite is experimental and you may encounter some unexpected behavior.
- The main difference is that imports will take a lot longer than usual. For example, the first `import scikitplot` can take roughly 10-20 s.
- If you notice problems, feel free to open an [scikit-plots issue tracker](https://github.com/scikit-plots/scikit-plots/issues/new/choose) about it.
- `micropip/piplite/pip` use `%pip` in JupyterLite instead of `pip` or `!pip`

</div>

In [ ]:
# JupyterLite-specific code
import numpy
import scipy
import matplotlib
import pandas
import joblib
import threadpoolctl
# https://pyodide.org/en/stable/usage/downloading-and-deploying.html#cdn
# PEP 503 'simple' Python package index for `micropip/piplite/pip`
# Installing the dependencies first, and then scikit-plots from Anaconda.org.
import piplite; await piplite.install(  # or micropip
   'scikit-plots==0.5.dev0',            # Download scikit-plots *pyodide_20XX_0_wasm32.whl
   index_urls=[
      'https://pypi.anaconda.org/scikit-plots-wheels-staging-nightly/simple',
      'https://pypi.anaconda.org/scientific-python-nightly-wheels/simple',
      'https://pypi.anaconda.org/pyodide/simple',
      'https://pypi.org/simple',
]); import sklearn; import scikitplot as sp; sp.show_versions();

In [ ]:
# This cell is added by Sphinx-Gallery
# It can be customized to whatever you like


# Persist and Reload a Rank-BM25 Index

.. currentmodule:: scikitplot.rank_bm25

A BM25 index can be published to a directory and loaded in a later process.
This example uses pre-tokenized input, stable document identities, and a
temporary directory so the gallery leaves no artifact behind.

The saved state contains derived lexical data such as terms, frequencies, IDF
values, document lengths, parameters, and ``doc_ids``.  Treat the artifact as
potentially sensitive even though it does not serialize the original document
objects.


In [ ]:
# Authors: The scikit-plots developers
# SPDX-License-Identifier: BSD-3-Clause

from __future__ import annotations

import tempfile
from pathlib import Path

from scikitplot.rank_bm25 import BM25Okapi

## 1. Build an index that can identify its results



In [ ]:
corpus = [
    ["python", "visualization"],
    ["lexical", "retrieval", "bm25"],
    ["vector", "retrieval", "annoy"],
]
doc_ids = ["visualization", "bm25", "annoy"]
index = BM25Okapi(corpus, doc_ids=doc_ids)

before = index.get_top_ids(["retrieval", "bm25"], n=3)
print("before save:", before)
print("build identity:", index.identity)

## 2. Publish into an isolated temporary directory
``save`` returns the concrete generation it wrote.  Callers normally keep the
publication root and let ``load`` follow its current-generation pointer.



In [ ]:
with tempfile.TemporaryDirectory(prefix="scikitplot-rank-bm25-") as tmp:
    publication = Path(tmp) / "index"
    generation = index.save(publication)

    print("publication root exists:", publication.is_dir())
    print("generation directory created:", generation.is_dir())

    restored = BM25Okapi.load(publication)
    after = restored.get_top_ids(["retrieval", "bm25"], n=3)

    print("after load:", after)
    print("identity preserved:", restored.identity == index.identity)
    print("document IDs preserved:", restored.doc_ids == index.doc_ids)

    assert restored.identity == index.identity
    assert restored.doc_ids == index.doc_ids
    assert after == before

## 3. What is intentionally absent from this example
A bare callable passed as ``tokenizer=`` cannot be persisted because it does
not carry a durable analyzer identity.  The implementation has a declaration
type for that purpose, but it is not currently exported from the public
``scikitplot.rank_bm25`` namespace.  Until that public contract is resolved,
the gallery keeps persistence examples on the supported pre-tokenized path
rather than teaching a private import.

.. tags::

   model-workflow: retrieval
   plot-type: text
   level: intermediate
   purpose: workflow



In [ ]:
# This is the last cell